[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/08_data_vis/08_1_Basic_Charts.ipynb)

# 08.1: Basic Charts

A table of numbers can answer a specific question, but it is hard to see the overall shape of a column or to spot an unusual value in a table of 51 rows. A chart shows those things at a glance. This notebook introduces **seaborn**, a Python library for making charts, and the basic pattern for making three kinds of chart. Notebook 08.2 then covers how to choose among them.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

We use FBI crime rates per 100,000 people for U.S. states from 1960 to 2012. As you may remember from the pair activity, this data has a `United States` row, so `places_2012` leaves it out. It has the 50 states and the District of Columbia for 2012.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Violent.All": "Violent", "Rates.Property.All": "Property"})
crime = crime[["State", "Year", "Population", "Violent", "Property"]]
places_2012 = crime[(crime["Year"] == 2012) & (crime["State"] != "United States")]
print(places_2012.shape)
places_2012.head(3)

`import seaborn as sns` loads seaborn under its usual short name. Seaborn is built on a library called matplotlib, which we also load as `plt`. We will use `sns` to draw and matplotlib for small adjustments.

## A first chart from two lists

A chart can start from plain lists. A **line plot** connects points in order, and here it shows a made-up count of zombies by year. `plt.show()` displays the chart.

In [ ]:
year_list = [1970, 1980, 1990, 2000, 2010, 2020, 2030]
zombie_list = [0, 0, 0, 0, 10, 20, 50]
sns.lineplot(x=year_list, y=zombie_list)
plt.show()

The line stays at 0 until 2000 and then climbs to 50. Giving seaborn lists works, but a chart with no labels leaves the reader guessing, and real data usually lives in a DataFrame.

## The pattern

Every chart in this notebook follows the same pattern: tell seaborn which DataFrame to use with `data=`, and which columns to use with `x=` and `y=`. Seaborn returns an object that we store as `ax`, which is short for axes and represents the chart. We then use `ax` to add a title and axis labels.

In [ ]:
ax = sns.histplot(data=places_2012, x="Violent")
ax.set_title("Violent crime rates by state, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("Number of states")
plt.show()

## Histograms: the shape of one numeric column

The chart above is a **histogram**. It divides the range of a numeric column into bins and draws a bar showing how many values fall into each bin. Most states fall between about 200 and 650, with the tallest bar near 300, and one bar sits far off by itself on the right. That is the District of Columbia at 1243.7, which you could have missed in a table. This is the long tail from notebook 06.3, in picture form.

The shape can change with the number of bins, and you can set it with `bins=`. With too few, the bars hide the detail. With too many, a small dataset breaks up into scattered bars. It is worth trying a few values.

In [ ]:
ax = sns.histplot(data=places_2012, x="Violent", bins=6)
ax.set_title("The same data with 6 bins")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("Number of states")
plt.show()

With only 6 bins the chart is blockier, but D.C. still sits apart on the right. To compare two groups, make two calls on the same chart. Using `alpha=` makes the bars partly see-through so that both groups show.

In [ ]:
states_only = places_2012[places_2012["State"] != "District of Columbia"]
small = states_only[states_only["Population"] < 5000000]
large = states_only[states_only["Population"] >= 5000000]
ax = sns.histplot(data=small, x="Violent", bins=10, alpha=0.5, label="Under 5 million")
sns.histplot(data=large, x="Violent", bins=10, alpha=0.5, label="5 million or more")
ax.set_title("Violent crime rates by state size, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("Number of states")
ax.legend()
plt.show()

The smaller states have their tallest bar near 250 and the larger states have theirs near 400, but the two groups overlap heavily. Each call chose its own bin edges, which is why the bars do not line up.

## Box plots: a quick summary of the same shape

A **box plot** draws the numbers from `.describe()`. The box runs from the 25th to the 75th percentile, and the line inside it is the median. The lines extending from the box, called whiskers, reach the rest of the typical values, and points beyond them are drawn individually as possible outliers.

In [ ]:
print(places_2012["Violent"].describe())
ax = sns.boxplot(data=places_2012, x="Violent")
ax.set_title("Violent crime rates by state, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
plt.show()

The single point far to the right is again the District of Columbia. Compare the numbers from `.describe()` with the picture, and you can match the 25%, 50% and 75% rows to the box. A box plot gives less detail about shape than a histogram, but it takes up less space.

Box plots are at their best when comparing groups, with one box per category. Using `y=` for the category makes the labels easier to read. Here are the four states that start with I, across all years of the data.

In [ ]:
i_states = crime[crime["State"].str.startswith("I")]
ax = sns.boxplot(data=i_states, x="Violent", y="State")
ax.set_title("Violent crime rates, 1960-2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("")
plt.show()

Illinois has the highest rates and the widest box of the four, and Idaho and Iowa have the lowest. Indiana has one point drawn on its own at the low end. Each box summarizes the years of data for one state.

## Bar charts: comparing categories

A **bar chart** shows one number for each category. The usual approach is to calculate the numbers yourself with `.groupby()` and `.reset_index()`, from notebook 07.4, and then draw them. We will use the Broadway data, since it has categories.

In [ ]:
broadway = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
mean_gross = broadway.groupby("Type")["Gross"].mean().reset_index()
mean_gross

In [ ]:
ax = sns.barplot(data=mean_gross, x="Type", y="Gross")
ax.set_title("Mean weekly gross by type of show")
ax.set_xlabel("Type of show")
ax.set_ylabel("Mean weekly gross (dollars)")
plt.show()

Musicals stand out at about 700000, compared with about 365000 for specials and about 326000 for plays. If you give `sns.barplot()` the full data with many rows per category, it calculates the mean for you and adds a thin line that shows the uncertainty in that mean. Calculating the numbers first makes it clear exactly what the bars show.

To count rows in each category instead, `sns.countplot()` does the counting for you.

In [ ]:
ax = sns.countplot(data=broadway, x="Type")
ax.set_title("Number of show-weeks by type")
ax.set_xlabel("Type of show")
ax.set_ylabel("Number of show-weeks")
plt.show()

Musicals have 22551 rows, plays 8406 and specials only 339. Using `y=` instead of `x=` makes either chart horizontal. `hue=` splits each bar by a second category, which is called a grouped bar chart. The next chart compares the four I states in two different years.

In [ ]:
two_years = i_states[i_states["Year"].isin([1990, 2012])]
ax = sns.barplot(data=two_years, x="State", y="Violent", hue="Year")
ax.set_title("Violent crime rates in 1990 and 2012")
ax.set_xlabel("")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()

All four states have shorter bars in 2012, and the drop in Illinois is by far the largest. Charts have a default size, and if you need a different one, call `plt.figure(figsize=(10, 6))` before the plotting call, where the numbers are the width and the height.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
df_start = df.copy()
mean_attendance = df.groupby("Type")["Attendance"].mean().sort_values()
type_counts = df["Type"].value_counts().sort_values()

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")


def check_chart(number, name, test, hint, y_label=True):
    """Check a chart stored in a variable: it has labels, and test(chart) is True."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
        return
    chart = globals()[name]
    if not hasattr(chart, "get_title"):
        print(f"Problem {number}: not yet. Store what the seaborn function returns in {name}.")
    elif not chart.get_title() or not chart.get_xlabel() or (y_label and not chart.get_ylabel()):
        print(f"Problem {number}: not yet. Give the chart a title and label its axes.")
    else:
        try:
            ok = bool(test(chart))
        except (AttributeError, IndexError, TypeError, ValueError):
            ok = False
        print(f"Problem {number}: correct." if ok else f"Problem {number}: not yet. {hint}")

### Problem 1
Make a histogram of `Attendance`, and store what seaborn returns in `attendance_hist`. Give it a title and label both axes.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check_chart(1, "attendance_hist", lambda c: len(c.patches) > 1 and c.get_xlabel() != "", "Use sns.histplot() with data=df and x=\"Attendance\".")

### Problem 2
Make another histogram of `Attendance` with exactly 20 bins, and store it in `attendance_20`. Give it a title and label both axes.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check_chart(2, "attendance_20", lambda c: len(c.patches) == 20, "Add bins=20.")

### Problem 3
Make a box plot of `Attendance` for each `Type`, with one box per type, and store it in `attendance_box`. Give it a title and label both axes.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check_chart(3, "attendance_box", lambda c: {t.get_text() for t in c.get_xticklabels() + c.get_yticklabels()} >= set(df_start["Type"].unique()), "Use sns.boxplot() with Type on one axis and Attendance on the other.")

### Problem 4
Make a bar chart of the mean `Attendance` for each `Type`, and store it in `attendance_bar`. Calculate the means with `.groupby()` and `.reset_index()` first. Give it a title and label both axes.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check_chart(4, "attendance_bar", lambda c: np.allclose(sorted(p.get_height() for p in c.patches if p.get_height() > 0), mean_attendance.to_list()), "The bar heights should be the mean attendance for each type, with vertical bars.")

### Problem 5
Make a count plot of `Type`, and store it in `type_count`. Give it a title and label both axes.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check_chart(5, "type_count", lambda c: np.allclose(sorted(p.get_height() for p in c.patches if p.get_height() > 0), type_counts.to_list()), "Use sns.countplot() with x=\"Type\".")

## What's next

You can now make a histogram, a box plot and a bar chart, with a title and labels. These are only useful if they fit the question you are asking, though. Notebook 08.2 adds scatter plots and regression lines, and it shows how to choose the right kind of chart for a question and how to say what a chart shows.